In [11]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations



In [12]:
def make_M_V(P, w):
    M = P @ np.diag(w) @ P.T
    return M, np.diag(P @ w) - M


In [13]:
#Q is quadratioc objective rather than autocorrelation matrix
def max_cut_from_Q(Q, 
                   seed: int,
                   num_layers: int,
                   scaling_constant: float,
                   step_size: float, 
                   max_trials: int,
                   threshold: float,
                  ):
    rng = np.random.default_rng(seed=seed)
    n_wires = Q.shape[0]
    triu_indices = np.triu_indices(Q.shape[0])
    edge_list = []
    for (u, v) in zip(triu_indices[0], triu_indices[1]):
        if u == v:
            continue
        elif Q[u][v] != 0.0:
            edge_list.append((u, v))
    num_edges = len(edge_list)
    print("Number of edges: ", num_edges)
    beta_params = scaling_constant * rng.random([n_wires, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    def U_B(beta):
        for wire in range(n_wires):
            qml.RX(2 * beta[wire], wires=wire)
    def U_C(gamma):
        edge_idx = 0
        for u, v in edge_list: 
            w = Q[u][v]
            qml.CNOT(wires=(u, v))
            qml.RZ(-w * gamma[edge_idx], wires=v)
            qml.CNOT(wires=(u, v))
            edge_idx += 1 
    dev = qml.device('default.qubit', wires=n_wires)
    @qml.qnode(dev)
    def circuit(gammas, betas):
        for wire in range(n_wires):
            qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return [qml.expval(qml.PauliZ(u) @ qml.PauliZ(v)) for u, v in edge_list]
    
    def objective(gammas, betas):
        expectation_values = circuit(gammas, betas)
        expectation_values = np.array(expectation_values)
        values = 0.0
        for u in range(n_wires):
            values += Q[u][u]
        edge_idx = 0
        for u, v in edge_list:
            w  = Q[u][v]
            values += 2 * expectation_values[edge_idx] * w
            edge_idx += 1
        return -values

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_list = [objective(gamma_params, beta_params)]
    params_list = [(gamma_params, beta_params)]
    for i in range(max_trials):
        (gamma_params, beta_params), current_obj = opt.step_and_cost(objective, gamma_params, beta_params)
        if (i + 1) % 50 == 0:
            print("Current obj: ", current_obj)
            if np.abs(current_obj - objective_list[-1]) <= threshold:
                objective_list.append(current_obj)
                params_list.append((gamma_params, beta_params))
                break
        objective_list.append(current_obj)
        params_list.append((gamma_params, beta_params))
    result = [objective_list, params_list]
    best_params = result[1][np.argmin(result[0])]
    best_qaoa_objective = np.min(result[0])
    
    #print("Best param: ", best_params)
    print("Best QAOA obj: ", best_qaoa_objective)
    
    dev_sampling = qml.device('default.qubit', wires=n_wires)
    @qml.set_shots(shots = 10000)
    @qml.qnode(dev_sampling)
    def sampling_circuit(gammas, betas):
        for wire in range(n_wires):
            qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return qml.sample()
    best_gammas, best_betas = best_params
    samples = sampling_circuit(best_gammas, best_betas)
    values = []
    formatted_samples = []
    for sample in samples:
        new_x = 2*np.array(sample)-1
        value = new_x@Q@new_x.transpose()
        formatted_samples.append(sample)
        values.append(value)
    print("Best objective:", np.max(values))
    print("Average objective:", np.mean(values))
    print("Best x:", formatted_samples[np.argmax(values)])
    return [np.mean(values), best_qaoa_objective, best_params] #np.max(values), formatted_samples[np.argmax(values)], 





In [14]:
def run_graph(idx, num_layers, run_type, graph_type):
    # if graph_type == "contact-high-school":
    #     if idx < 10: 
    #         hypergraph_name = f"contact-high-school_0{idx}"
    #     else:
    #         hypergraph_name = f"contact-high-school_{idx}"
    # elif graph_type == "email-Enron":
    #     hypergraph_name = f"email-Enron_{idx}"
    # elif graph_type == "congress-bills":
    #     if idx < 10:
    #         hypergraph_name = f"congress-bills_0{idx}"
    #     else:
    #         hypergraph_name = f"congress-bills_{idx}"
    if idx < 10:
        hypergraph_name = f"0{idx}"
    else:
        hypergraph_name = f"{idx}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    M, V = make_M_V(P, w)
    Vlist = []
    Q = V
    Vlist = [w[i] * (np.diag(P[:, i]) - P[:, i].reshape(P.shape[0], 1) * P[:, i].reshape(1, P.shape[0])) for i in range(P.shape[1])]
    if run_type == "max":
        step_size = 5e-2
    elif run_type == "fair":
        step_size = 5e-2
    threshold = 1e-8
    max_trials = 300
    scaling_constant = 5e-2
    seed = 42
    if run_type == "max":
        results  =  max_cut_from_Q(Q=Q,
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best average max:", results[0])
    elif run_type == "fair":
        results  =  fair_cut_from_Q(Vlist = Vlist, #
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best fair sampling:", results[0])
    return results


In [15]:
results = run_graph(
    idx=1,
    num_layers=1,
    run_type="max",
    graph_type="poisson_16_12_3"
)

Name:  01
Number of edges:  43
Current obj:  -13.140008297894417
Current obj:  -13.825591805612621
Current obj:  -14.123585378257804
Current obj:  -14.385076897351968
Current obj:  -14.569653349549133
Current obj:  -14.742119589227748
Best QAOA obj:  -14.742119589227748
Best objective: 15.142222222222221
Average objective: 14.743250555555552
Best x: [1 0 0 1 1 0 1 0 1 1 1 0]
Best QAOA obj:  -14.742119589227748
Best average max: 14.743250555555552


In [16]:
from math import comb
#can update to max_cut_from_Q_hwpm(Q_cost,Q_mixer

#hamming weight preserving mixer
def max_cut_from_Q_hwpm(Q,
                   seed: int,
                   num_layers: int,
                   scaling_constant: float,
                   step_size: float, 
                   max_trials: int,
                   threshold: float,
                  ):
    rng = np.random.default_rng(seed=seed)
    n_wires = Q.shape[0]
    triu_indices = np.triu_indices(Q.shape[0])
    edge_list = []
    for (u, v) in zip(triu_indices[0], triu_indices[1]):
        if u == v:
            continue
        elif Q[u][v] != 0.0:
            edge_list.append((u, v))
    num_edges = len(edge_list)
    print("Number of edges: ", num_edges)
    
    #n_wires change to num_edges because we need theta for each edge(previously was #of wires)
    beta_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    gamma_params = scaling_constant * rng.random([num_edges, num_layers], requires_grad=True)
    #nx graph with edge list(path) 
    def U_B(beta):
        edge_idx=0
        for u, v in edge_list:
            theta = beta[edge_idx] #theta for each edge
            qml.S(wires=u) #s gate
            #correct wire ordering??
            qml.SingleExcitationMinus(-2*theta,wires=[u,v])
            qml.adjoint(qml.S)(wires=u) #s adjoint
            edge_idx+=1
        
        #for wire in range(n_wires):
        #    qml.RX(2 * beta[wire], wires=wire)
    def U_C(gamma):
        edge_idx = 0
        for u, v in edge_list: 
            w = Q[u][v]
            qml.CNOT(wires=(u, v))
            qml.RZ(-w * gamma[edge_idx], wires=v)
            qml.CNOT(wires=(u, v))
            edge_idx += 1 
    dev = qml.device('default.qubit', wires=n_wires)

    #initial state equal hamming weight of 000...111
    #target_hamming_weight = n_wires // 2
    #initial_state = np.array([1] * target_hamming_weight+ [0] * (n_wires - target_hamming_weight),requires_grad=False)

    def make_dicke_state(n, k):
        #generally k will be half of n for balanced partition
        if k < 0 or k > n:
            raise ValueError("k must satisfy 0 <= k <= n")
    
        state = np.zeros(2**n, dtype=complex)
    
        # All choices of k wires that contain a 1
        weight_k_states = list(combinations(range(n), k))
        # equal emplitude for each state
        amplitude = 1 / np.sqrt(len(weight_k_states))

        # will have sets of bits now ex: for n=4 and k=2 --> (0,1)(0,2),(0,3),(1,2),(1,3),(2,3) --> set a one at each of these index
        #(0,1) -> |1100> -> index 12
        #index 11 skipped so that amplitude stays at 0
        #(0,2) -> |1010> -> index 10
        
        for one_positions in weight_k_states:
            basis_index=0
            #wire index to statevector index
            for wire in one_positions:
                basis_index+=2**(n-1-wire)
            #basis_index = sum(2 ** (n - 1 - wire) for wire in one_positions)
            state[basis_index] = amplitude

        

        d_state = np.array(state, requires_grad=False)
        if (comb(n,k)!=np.count_nonzero(d_state)):
            raise ValueError("amplitude count incorrect")
    
        return d_state

    #hamming weight will be n/2 for even number of 0's and 1's
    target_weight = n_wires // 2
    initial_state = make_dicke_state(n_wires, target_weight)

    #confirm number of non-zero amps and then confirm they sum to one
    print("Number of nonzero amplitudes:", np.count_nonzero(initial_state))
    print("Predicted number of non-zero amplitudes: ", comb(n_wires, target_weight))
    print("State norm:", np.sum(np.abs(initial_state) ** 2))
        
    @qml.qnode(dev)
    def circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        #for wire in range(n_wires):
        #    qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return [qml.expval(qml.PauliZ(u) @ qml.PauliZ(v)) for u, v in edge_list]
    
    def objective(gammas, betas):
        expectation_values = circuit(gammas, betas)
        expectation_values = np.array(expectation_values)
        values = 0.0
        for u in range(n_wires):
            values += Q[u][u]
        edge_idx = 0
        for u, v in edge_list:
            w  = Q[u][v]
            values += 2 * expectation_values[edge_idx] * w
            edge_idx += 1
        return -values

    opt = qml.AdamOptimizer(stepsize=step_size)
    objective_list = [objective(gamma_params, beta_params)]
    params_list = [(gamma_params, beta_params)]
    for i in range(max_trials):
        (gamma_params, beta_params), current_obj = opt.step_and_cost(objective, gamma_params, beta_params)
        if (i + 1) % 50 == 0:
            print("Current obj: ", current_obj)
            if np.abs(current_obj - objective_list[-1]) <= threshold:
                objective_list.append(current_obj)
                params_list.append((gamma_params, beta_params))
                break
        objective_list.append(current_obj)
        params_list.append((gamma_params, beta_params))
    result = [objective_list, params_list]
    best_params = result[1][np.argmin(result[0])]
    best_qaoa_objective = np.min(result[0])
    
    print("Best param: ", best_params)
    print("Best QAOA obj: ", best_qaoa_objective)
    
    dev_sampling = qml.device('default.qubit', wires=n_wires)
    @qml.set_shots(shots = 10000)
    @qml.qnode(dev_sampling)
    def sampling_circuit(gammas, betas):
        qml.StatePrep(initial_state, wires=range(n_wires))
        #for wire in range(n_wires):
        #    qml.Hadamard(wires=wire)
        for k in range(num_layers):
            U_C(gammas[:, k])
            U_B(betas[:, k])
        return qml.sample()
    best_gammas, best_betas = best_params
    samples = sampling_circuit(best_gammas, best_betas)
    ###delete later-chech\k
    #confirm each sample has hamming weight of n/2
    sample_weights = np.sum(samples, axis=1)
    print("Unique sampled Hamming weights:", np.unique(sample_weights))
    print("length of samples: ",len(sample_weights))
    ###
    values = []
    formatted_samples = []
    for sample in samples:
        new_x = 2*np.array(sample)-1
        value = new_x@Q@new_x.transpose()
        formatted_samples.append(sample)
        values.append(value)
    print("Best objective:", np.max(values))
    print("Average objective:", np.mean(values))
    print("Best x:", formatted_samples[np.argmax(values)])
    return [np.mean(values), best_qaoa_objective, best_params] #np.max(values), formatted_samples[np.argmax(values)], 





In [17]:
def run_graph_hwpm(idx, num_layers, run_type, graph_type):
    # if graph_type == "contact-high-school":
    #     if idx < 10: 
    #         hypergraph_name = f"contact-high-school_0{idx}"
    #     else:
    #         hypergraph_name = f"contact-high-school_{idx}"
    # elif graph_type == "email-Enron":
    #     hypergraph_name = f"email-Enron_{idx}"
    # elif graph_type == "congress-bills":
    #     if idx < 10:
    #         hypergraph_name = f"congress-bills_0{idx}"
    #     else:
    #         hypergraph_name = f"congress-bills_{idx}"
    if idx < 10:
        hypergraph_name = f"0{idx}"
    else:
        hypergraph_name = f"{idx}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    M, V = make_M_V(P, w)
    Vlist = []
    Q = V
    print(Q)
    #create network X path graph w same number of nodes
    Vlist = [w[i] * (np.diag(P[:, i]) - P[:, i].reshape(P.shape[0], 1) * P[:, i].reshape(1, P.shape[0])) for i in range(P.shape[1])]
    if run_type == "max":
        step_size = 5e-2
    elif run_type == "fair":
        step_size = 5e-2
    threshold = 1e-8
    max_trials = 300
    scaling_constant = 5e-2
    seed = 42
    if run_type == "max":
        results  =  max_cut_from_Q_hwpm(Q=Q,
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best average max:", results[0])
    elif run_type == "fair":
        results  =  fair_cut_from_Q(Vlist = Vlist, #
                                    seed = seed,
                                    num_layers = num_layers,
                                    scaling_constant = scaling_constant,
                                    step_size = step_size, 
                                    max_trials = max_trials,
                                    threshold = threshold)
        print("Best QAOA obj: ", results[1])
        print("Best fair sampling:", results[0])
    return results


In [18]:
results = run_graph_hwpm(
    idx=1,
    num_layers=1,
    run_type="max",
    graph_type="poisson_16_12_3"
)
#check if initial state is a ground state for the mixer
#cis(\theta)

Name:  01
[[ 0.82666667  0.         -0.04        0.          0.         -0.26222222
   0.          0.         -0.33333333  0.         -0.04       -0.15111111]
 [ 0.          0.36111111 -0.02777778  0.         -0.13888889 -0.02777778
  -0.02777778  0.         -0.11111111 -0.02777778  0.          0.        ]
 [-0.04       -0.02777778  1.05611111 -0.125      -0.13888889 -0.06777778
  -0.06777778 -0.165      -0.11111111 -0.13027778 -0.1425     -0.04      ]
 [ 0.          0.         -0.125       0.625       0.         -0.25
   0.         -0.125       0.         -0.0625     -0.0625      0.        ]
 [ 0.         -0.13888889 -0.13888889  0.          1.05555556 -0.13888889
  -0.02777778  0.         -0.33333333 -0.02777778  0.         -0.25      ]
 [-0.26222222 -0.02777778 -0.06777778 -0.25       -0.13888889  1.87527778
  -0.09027778 -0.0625     -0.58333333 -0.13888889 -0.21361111 -0.04      ]
 [ 0.         -0.02777778 -0.06777778  0.         -0.02777778 -0.09027778
   0.48638889 -0.1025      0